# WHO Global Health Observatory: Data Exploration

Each exploration notebook for a data source should work through the key steps:
1. Loading the data
2. Determining the observation units and the variables of interest
3. Locating and handling missing data
4. Transforming/adding new variables
5. Creating and saving clean subsets
6. Identifying the key dimensions in the dataset (temporal, spatial, categorical)
7. Listing the key questions relating to the overarching research questions
8. Carrying out descriptive analysis of each of the key variables and relevant combinations

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import math
import plotly
import warnings
warnings.simplefilter('ignore')
import numpy as np
from scipy import stats

## 1. Loading the Data

The WHO Global Health Observatory provides two datasets:
- **`who_hale_at_birth.csv`** — Healthy Life Expectancy (HALE) at birth: the expected number of years lived in full health
- **`who_le_at_birth.csv`** — Life Expectancy (LE) at birth: total expected years lived regardless of health state

Both datasets cover 185 countries from 2000 to 2021, broken down by sex (Male, Female, Both sexes).

In [ ]:
hale_raw = pd.read_csv('../data/raw/who_hale_at_birth.csv')
le_raw   = pd.read_csv('../data/raw/who_le_at_birth.csv')

print(f'HALE shape: {hale_raw.shape}')
print(f'LE shape:   {le_raw.shape}')
print(f'\nColumns: {hale_raw.columns.tolist()}')
hale_raw.head(3)

## 2. Observation Units and Variables of Interest

**Observation unit:** Country × Year × Sex combination

**Key variables retained from the raw data:**
| Column | Meaning |
|---|---|
| `Location` | Country name |
| `SpatialDimValueCode` | ISO3 country code |
| `ParentLocation` | WHO region |
| `Period` | Year (2000–2021) |
| `Dim1` | Sex (Male / Female / Both sexes) |
| `FactValueNumeric` | The HALE or LE value in years |
| `FactValueNumericLow` | Lower uncertainty bound |
| `FactValueNumericHigh` | Upper uncertainty bound |

In [ ]:
keep_cols = ['SpatialDimValueCode', 'Location', 'ParentLocation', 'Period', 'Dim1',
             'FactValueNumeric', 'FactValueNumericLow', 'FactValueNumericHigh']

hale = hale_raw[keep_cols].rename(columns={
    'SpatialDimValueCode': 'country_code',
    'Location': 'country',
    'ParentLocation': 'region',
    'Period': 'year',
    'Dim1': 'sex',
    'FactValueNumeric': 'hale',
    'FactValueNumericLow': 'hale_low',
    'FactValueNumericHigh': 'hale_high'
})

le = le_raw[keep_cols].rename(columns={
    'SpatialDimValueCode': 'country_code',
    'Location': 'country',
    'ParentLocation': 'region',
    'Period': 'year',
    'Dim1': 'sex',
    'FactValueNumeric': 'le',
    'FactValueNumericLow': 'le_low',
    'FactValueNumericHigh': 'le_high'
})

print(f'Countries: {hale["country"].nunique()}')
print(f'Years: {hale["year"].min()} – {hale["year"].max()}')
print(f'Sex categories: {hale["sex"].unique()}')
print(f'WHO regions: {hale["region"].unique()}')

## 3. Locating and Handling Missing Data

In [ ]:
print('=== HALE missing values ===')
print(hale.isna().sum())
print('\n=== LE missing values ===')
print(le.isna().sum())

Both datasets are complete — no missing values in any column. The WHO estimates uncertainty bounds for every country-year-sex combination, so the data is already well-structured.

## 4. Transforming and Adding New Variables

The key derived variable is the **HALE gap** — the difference between life expectancy and healthy life expectancy. This represents the average number of years a person lives in poor health or disability. A smaller gap means a healthier population relative to how long they live.

In [ ]:
# Merge HALE and LE on country, year, sex
who = pd.merge(
    hale,
    le[['country_code', 'year', 'sex', 'le', 'le_low', 'le_high']],
    on=['country_code', 'year', 'sex'],
    how='inner'
)

# HALE gap: years lived in poor health = LE - HALE
who['hale_gap'] = who['le'] - who['hale']

# HALE ratio: proportion of life spent in good health
who['hale_ratio'] = who['hale'] / who['le']

print(f'Merged dataset shape: {who.shape}')
who.head(5)

## 5. Creating and Saving Clean Subsets

In [ ]:
import os
os.makedirs('../data/clean', exist_ok=True)

# Full cleaned dataset
who.to_csv('../data/clean/who_clean.csv', index=False)
print(f'Saved who_clean.csv — {who.shape[0]} rows')

# Both-sexes only, latest year (2021) — for cross-sectional analysis
who_latest = who[(who['sex'] == 'Both sexes') & (who['year'] == 2021)].copy()
who_latest.to_csv('../data/clean/who_latest.csv', index=False)
print(f'Saved who_latest.csv — {who_latest.shape[0]} rows')

# Both-sexes time series — for trend analysis
who_both = who[who['sex'] == 'Both sexes'].copy()
who_both.to_csv('../data/clean/who_both_sexes.csv', index=False)
print(f'Saved who_both_sexes.csv — {who_both.shape[0]} rows')

## 6. Key Dimensions in the Dataset

- **Temporal:** 2000 to 2021, annual — 22 years of global health data
- **Spatial:** 185 countries across all WHO regions
- **Categorical (sex):** Male, Female, Both sexes
- **Categorical (region):** Africa, Americas, Eastern Mediterranean, Europe, South-East Asia, Western Pacific
- **Key limitation:** Data only goes to 2021 — no COVID-era recovery data. The 2020–2021 dip in many countries reflects pandemic mortality.

In [ ]:
# Coverage: countries per region
region_counts = who_latest.groupby('region')['country'].count().sort_values(ascending=False)
print('Countries per WHO region (2021 data):')
print(region_counts.to_string())

## 7. Key Questions Relating to the Research Questions

The overarching project asks: **which countries/cities offer the best value for living** — high quality of life at a reasonable cost?

From the WHO data, the relevant questions are:

1. **Which countries have the highest HALE at birth?** — Identifies countries where people not only live long but live healthily.
2. **What is the HALE gap (LE − HALE) across countries?** — Countries with a small gap have efficient healthcare; people spend fewer years sick.
3. **How do HALE and LE differ between sexes across regions?** — Understand whether the female longevity advantage is universal.
4. **What are the regional trends in HALE and LE from 2000 to 2021?** — Captures convergence/divergence between developing and developed regions.
5. **Does WHO LE align with World Bank LE for the same countries?** — Data validation before merging sources.
6. **Which countries saw the biggest HALE improvements over 2000–2021?** — Reveals which nations made the fastest health gains.

## 8. Descriptive Analysis

### Q1: Which countries have the highest and lowest HALE at birth (2021)?

In [ ]:
print('Top 15 countries by HALE (Both sexes, 2021):')
print(who_latest.nlargest(15, 'hale')[['country', 'region', 'hale', 'le', 'hale_gap']]
      .to_string(index=False))

print('\nBottom 15 countries by HALE (Both sexes, 2021):')
print(who_latest.nsmallest(15, 'hale')[['country', 'region', 'hale', 'le', 'hale_gap']]
      .to_string(index=False))

Singapore, Japan, and South Korea top the HALE rankings — all high-income East Asian countries with strong healthcare systems and diets. The bottom countries are concentrated in sub-Saharan Africa, where infectious disease burden (HIV, malaria) and fragile health infrastructure pull down healthy life years.

In [ ]:
top15 = who_latest.nlargest(15, 'hale').sort_values('hale')
bot15 = who_latest.nsmallest(15, 'hale').sort_values('hale', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].barh(top15['country'], top15['hale'], color='#0D9488')
axes[0].set_title('Top 15 Countries — HALE at Birth (2021)', fontsize=13)
axes[0].set_xlabel('HALE (years)')
axes[0].set_xlim(60, 80)

axes[1].barh(bot15['country'], bot15['hale'], color='#DC2626')
axes[1].set_title('Bottom 15 Countries — HALE at Birth (2021)', fontsize=13)
axes[1].set_xlabel('HALE (years)')

plt.tight_layout()
plt.show()

### Q2: What is the HALE gap (years lost to disease/disability) across countries?

A small HALE gap means people spend a higher proportion of their lives in good health — arguably more important than raw life expectancy for quality-of-life comparisons.

In [ ]:
print('HALE gap summary (Both sexes, 2021):')
print(who_latest['hale_gap'].describe().round(2))

print('\nSmallest HALE gap (healthiest relative to longevity):')
print(who_latest.nsmallest(10, 'hale_gap')[['country', 'region', 'le', 'hale', 'hale_gap']]
      .to_string(index=False))

print('\nLargest HALE gap (most years lived in poor health):')
print(who_latest.nlargest(10, 'hale_gap')[['country', 'region', 'le', 'hale', 'hale_gap']]
      .to_string(index=False))

In [ ]:
fig = px.scatter(
    who_latest,
    x='le',
    y='hale',
    color='region',
    hover_name='country',
    size='hale_gap',
    size_max=20,
    title='Life Expectancy vs HALE by Country (2021) — bubble size = HALE gap',
    labels={'le': 'Life Expectancy (years)', 'hale': 'Healthy Life Expectancy — HALE (years)'}
)
# Add diagonal reference line (HALE = LE, i.e. zero gap)
mn, mx = who_latest['le'].min(), who_latest['le'].max()
fig.add_shape(type='line', x0=mn, y0=mn, x1=mx, y1=mx,
              line=dict(dash='dash', color='grey', width=1))
fig.update_layout(plot_bgcolor='white', paper_bgcolor='white')
fig.show()

All countries fall below the diagonal (HALE < LE by definition). Countries in the upper-left area (high LE, modest HALE) spend many years in poor health. The European and Western Pacific clusters are tightly grouped with small gaps, while African countries show both lower HALE and wider gaps.

### Q3: How do HALE and LE differ between sexes across regions?

In [ ]:
sex_region = (
    who[(who['year'] == 2021) & (who['sex'] != 'Both sexes')]
    .groupby(['region', 'sex'])[['hale', 'le', 'hale_gap']]
    .mean()
    .round(2)
    .reset_index()
)
print('Average HALE and LE by region and sex (2021):')
print(sex_region.to_string(index=False))

In [ ]:
fig = px.bar(
    sex_region,
    x='region',
    y='hale',
    color='sex',
    barmode='group',
    color_discrete_map={'Male': '#3B82F6', 'Female': '#EC4899'},
    title='Average HALE by WHO Region and Sex (2021)',
    labels={'hale': 'HALE (years)', 'region': 'WHO Region'}
)
fig.update_layout(plot_bgcolor='white', paper_bgcolor='white', xaxis_tickangle=-20)
fig.show()

In [ ]:
# Check whether females have a larger HALE gap (more years in poor health) despite living longer
who_2021 = who[(who['year'] == 2021) & (who['sex'] != 'Both sexes')].copy()

gap_by_sex = who_2021.groupby('sex')[['le', 'hale', 'hale_gap']].mean().round(2)
print('Global averages by sex (2021):')
print(gap_by_sex)

Women live longer than men in every region, but they also tend to have a **larger HALE gap** — meaning more of their extra years are spent in poor health. This is a consistent global pattern: the female longevity advantage does not translate proportionally into additional healthy years.

### Q4: Regional trends in HALE and LE from 2000 to 2021

In [ ]:
trend = (
    who[who['sex'] == 'Both sexes']
    .groupby(['year', 'region'])[['hale', 'le']]
    .mean()
    .reset_index()
)

fig = px.line(
    trend,
    x='year',
    y='hale',
    color='region',
    markers=True,
    title='Average HALE at Birth by WHO Region (2000–2021)',
    labels={'hale': 'HALE (years)', 'year': 'Year', 'region': 'WHO Region'}
)
fig.update_layout(plot_bgcolor='white', paper_bgcolor='white')
fig.show()

In [ ]:
fig2 = px.line(
    trend,
    x='year',
    y='le',
    color='region',
    markers=True,
    title='Average Life Expectancy at Birth by WHO Region (2000–2021)',
    labels={'le': 'Life Expectancy (years)', 'year': 'Year', 'region': 'WHO Region'}
)
fig2.update_layout(plot_bgcolor='white', paper_bgcolor='white')
fig2.show()

Africa has made the steepest gains over this period — driven by antiretroviral rollout reducing HIV mortality. Europe and the Western Pacific remain the healthiest regions. The visible dip in 2020–2021 across all regions captures COVID-19 mortality. Note that regions have been converging, but a large gap between Africa and the rest remains.

### Q5: Does WHO Life Expectancy align with World Bank Life Expectancy?

Both sources report life expectancy at birth. Validating their agreement before merging datasets is essential.

In [ ]:
wb = pd.read_csv('../data/raw/worldbank_wdi_raw.csv', na_values=['..'])

# Reshape WB to long format
year_cols = [c for c in wb.columns if 'YR' in str(c)]
wb_long = wb.melt(
    id_vars=['Country Name', 'Country Code', 'Series Name'],
    value_vars=year_cols,
    var_name='year_raw',
    value_name='value'
)
wb_long['year'] = wb_long['year_raw'].str.extract(r'(\d{4})').astype(int)
wb_le = wb_long[wb_long['Series Name'] == 'Life expectancy at birth, total (years)'][['Country Code','year','value']].rename(
    columns={'Country Code':'country_code','value':'le_wb'}
).dropna()

# WHO both-sexes LE
who_le = who[who['sex'] == 'Both sexes'][['country_code','year','le']].rename(columns={'le':'le_who'})

# Merge
compare = pd.merge(wb_le, who_le, on=['country_code','year'])
print(f'Matched rows: {len(compare)}')

r, p = stats.pearsonr(compare['le_wb'], compare['le_who'])
print(f'Pearson r = {r:.4f},  p = {p:.2e}')

compare['diff'] = (compare['le_wb'] - compare['le_who']).abs()
print(f'\nMean absolute difference: {compare["diff"].mean():.2f} years')
print(f'Max absolute difference:  {compare["diff"].max():.2f} years')

print('\nLargest discrepancies:')
print(compare.nlargest(10,'diff')[['country_code','year','le_wb','le_who','diff']].to_string(index=False))

In [ ]:
fig = px.scatter(
    compare.sample(min(2000, len(compare)), random_state=42),
    x='le_wb',
    y='le_who',
    opacity=0.4,
    title=f'World Bank vs WHO Life Expectancy (r = {r:.3f})',
    labels={'le_wb': 'World Bank LE (years)', 'le_who': 'WHO LE (years)'}
)
mn = min(compare['le_wb'].min(), compare['le_who'].min())
mx = max(compare['le_wb'].max(), compare['le_who'].max())
fig.add_shape(type='line', x0=mn, y0=mn, x1=mx, y1=mx,
              line=dict(dash='dash', color='red', width=1))
fig.update_layout(plot_bgcolor='white', paper_bgcolor='white')
fig.show()

The two sources agree strongly for most countries (high r), but outliers exist — particularly small Gulf states (Qatar, Bahrain) and some island nations. The World Bank figures for these countries appear to include migrant worker populations differently from WHO estimates. For the merged analysis, **WHO LE is preferred** as the authoritative health source; World Bank data will supply economic indicators.

### Q6: Which countries made the biggest HALE improvements from 2000 to 2021?

In [ ]:
who_both = who[who['sex'] == 'Both sexes'].copy()

hale_2000 = who_both[who_both['year'] == 2000][['country','region','hale']].rename(columns={'hale':'hale_2000'})
hale_2021 = who_both[who_both['year'] == 2021][['country','hale']].rename(columns={'hale':'hale_2021'})

hale_change = pd.merge(hale_2000, hale_2021, on='country')
hale_change['change'] = hale_change['hale_2021'] - hale_change['hale_2000']

print('Top 15 countries: largest HALE gain 2000–2021:')
print(hale_change.nlargest(15,'change')[['country','region','hale_2000','hale_2021','change']].to_string(index=False))

print('\nBottom 5 countries: HALE declined or stagnated:')
print(hale_change.nsmallest(5,'change')[['country','region','hale_2000','hale_2021','change']].to_string(index=False))

In [ ]:
top_gains = hale_change.nlargest(20, 'change').sort_values('change')

fig = px.bar(
    top_gains,
    x='change',
    y='country',
    color='region',
    orientation='h',
    title='Top 20 Countries: HALE Gain 2000–2021 (Both sexes)',
    labels={'change': 'HALE increase (years)', 'country': 'Country'}
)
fig.update_layout(plot_bgcolor='white', paper_bgcolor='white', height=600)
fig.show()

Sub-Saharan African countries dominate the top gainers list — Rwanda, Ethiopia, and Malawi all added over 15 years of healthy life expectancy since 2000. This reflects the dramatic scale-up of HIV/AIDS treatment and child survival interventions. These countries started from a very low base; their absolute HALE levels in 2021 are still well below the global median.

### Global Map: HALE at Birth (2021)

In [ ]:
fig = px.choropleth(
    who_latest,
    locations='country_code',
    color='hale',
    hover_name='country',
    color_continuous_scale='RdYlGn',
    title='Healthy Life Expectancy (HALE) at Birth — 2021 (Both sexes)',
    labels={'hale': 'HALE (years)'}
)
fig.update_layout(geo=dict(showframe=False, showcoastlines=True))
fig.show()

## Summary

| Finding | Implication for Value-for-Living Analysis |
|---|---|
| East Asian countries (Japan, Singapore, Korea) lead on HALE | High health quality, but also high cost — likely not "value" sweet spots |
| African countries have large HALE gaps despite low LE | Health infrastructure deficit; unlikely to score well on quality-adjusted living |
| European region has small HALE gaps (efficient health years) | Strong candidates for value — decent health, potentially moderate cost |
| WHO and World Bank LE correlate well, with Gulf state exceptions | Safe to merge data sources, but flag Gulf countries |
| Africa made largest HALE gains 2000–2021 | Convergence happening, but absolute levels still low |
| Women have larger HALE gaps than men everywhere | Gender adjustment may be needed in quality-of-life scores |